# 1. STT(Speech to Text)

## 1) OpenAI API

In [1]:
from dotenv import load_dotenv 
load_dotenv()

True

In [2]:
from openai import OpenAI 

client = OpenAI()

In [4]:
# https://developers.openai.com/api/docs/guides/audio
with open("./audio/my_voice.mp3", "rb") as f:
    result = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",
        file=f,
        language="ko"
    )

print(result)

Transcription(text='안녕하세요. 개그우먼 이수지입니다. 반갑습니다.', logprobs=None, usage=UsageTokens(input_tokens=53, output_tokens=18, total_tokens=71, type='tokens', input_token_details=UsageTokensInputTokenDetails(audio_tokens=52, text_tokens=1)))


### 실시간 음성 변환

In [5]:
# 라이브러리 설치 uv add pyaudio speechrecognition pydub
import speech_recognition as sr 

r = sr.Recognizer()

with sr.Microphone() as source:
    print("말해주세요.")
    r.adjust_for_ambient_noise(source) # 주변 소음 조정
    # STEP 1. 마이크 입력 받기
    audio = r.listen(source)
    # STEP 2. 텍스트 변환
    print("인식 중입니다.....")
    text = r.recognize_openai(audio)
    print(f"인식된 텍스트: {text}")
    # STEP 3. 오디오 저장
    audio_file = audio.get_wav_data()
    with open("./audio/real_voice.wav", "wb") as f:
        f.write(audio_file)
    print("목소리 저장 완료!")

말해주세요.
인식 중입니다.....
인식된 텍스트: 안녕하세요
목소리 저장 완료!


In [ ]:
# 오디오 백그라운드 실행
from pydub import AudioSegment 
from pydub.playback import play 

# sound = AudioSegment.from_mp3("./audio/my_voice.mp3")
sound = AudioSegment.from_wav("./audio/real_voice.wav")
play(sound)


## 2) 로컬 Whisper 모델 

In [15]:
# whisper github 검색 
# 라이브러리 설치 uv add openai-whisper
import whisper 

# 모델 
model = whisper.load_model("base")

# 텍스트 변환
result = model.transcribe(
    "./audio/my_voice.mp3",
    language="ko",
    fp16=True  # CPU 사용할거라면 False 
)

print(result)
print(f"인식된 텍스트: {result["text"]}")

{'text': ' 안녕하세요. 개그우먼 이수지입니다. 반갑습니다.', 'segments': [{'id': 0, 'seek': 0, 'start': 0.0, 'end': 3.68, 'text': ' 안녕하세요. 개그우먼 이수지입니다.', 'tokens': [50364, 19289, 13, 14552, 22069, 22471, 48150, 4329, 230, 246, 1831, 7416, 13, 50548], 'temperature': 0.0, 'avg_logprob': -0.4344204493931362, 'compression_ratio': 0.9166666666666666, 'no_speech_prob': 0.03106408752501011}, {'id': 1, 'seek': 0, 'start': 3.68, 'end': 5.18, 'text': ' 반갑습니다.', 'tokens': [50548, 16396, 27358, 3115, 13, 50623], 'temperature': 0.0, 'avg_logprob': -0.4344204493931362, 'compression_ratio': 0.9166666666666666, 'no_speech_prob': 0.03106408752501011}], 'language': 'ko'}
인식된 텍스트:  안녕하세요. 개그우먼 이수지입니다. 반갑습니다.


## 3) Faster-Whisper 모델

In [16]:
# faster-whisper github 검색
# 라이브러리 설치 uv add faster-whisper 
from faster_whisper import WhisperModel 

# 모델 
model = WhisperModel(
    "base", 
    device="cuda", 
    compute_type="float16"
)

# 텍스트 변환 
segments, info = model.transcribe(
    "./audio/my_voice.mp3",
    language="ko",
    beam_size=5 # 높을수록 정확하지만 느려진다.(default=5)
)

print(segments)
full_text = ""
for seg in segments:
    print(f"[{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text}")
    full_text += seg.text
print(f"인식된 텍스트: {full_text}")
print(f"감지된 언어: {info.language} (확률: {info.language_probability:.0%})")

<generator object WhisperModel.generate_segments at 0x00000251B824C630>
[0.00s -> 3.68s]  안녕하세요. 개그우먼 이수지입니다.
[3.68s -> 5.18s]  반갑습니다.
인식된 텍스트:  안녕하세요. 개그우먼 이수지입니다. 반갑습니다.
감지된 언어: ko (확률: 100%)


# 2.  TTS(Text to Speech)

## 1) Openai API

In [ ]:
# https://www.openai.fm/ 목소리 탐험

In [17]:
from openai import OpenAI 

client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input="안녕하세요. 반가워요",
    speed=0.3, # 0.25 ~ 4.0
    instructions="슬픈 목소리로 말해줘"# "밝고 자신감 있는 목소리로 말해줘"
) as response:
    response.stream_to_file("./audio/speech.mp3")

In [18]:
# 오디오 백그라운드 실행
from pydub import AudioSegment 
from pydub.playback import play 

sound = AudioSegment.from_mp3("./audio/speech.mp3")
play(sound)


## 2) ElevenLabs API

In [ ]:
# 로그인 https://elevenlabs.io/ 
# 라이브러리 설치 uv add elevenlabs

In [1]:
from dotenv import load_dotenv 
load_dotenv()

True

In [2]:
import os 
from elevenlabs.client import ElevenLabs

client = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))

In [3]:
# 음성 목록 가져오기
response = client.voices.search()

for voice in response.voices:
    print(voice)

voice_id='xi3rF0t7dg7uN2M0WUhr' name='Yuna' samples=None category='professional' fine_tuning=FineTuningResponse(is_allowed_to_fine_tune=True, state={'eleven_turbo_v2_5': 'fine_tuned', 'eleven_v2_5_flash': 'fine_tuned', 'eleven_multilingual_v2': 'fine_tuned', 'eleven_multilingual_sts_v2': 'fine_tuned', 'eleven_flash_v2_5': 'fine_tuned'}, verification_failures=[], verification_attempts_count=0, manual_verification_requested=False, language='ko', progress={}, message={'eleven_turbo_v2_5': '', 'eleven_v2_5_flash': '', 'eleven_multilingual_v2': '', 'eleven_multilingual_sts_v2': '', 'eleven_flash_v2_5': ''}, dataset_duration_seconds=None, verification_attempts=None, slice_ids=None, manual_verification=None, max_verification_attempts=0, next_max_verification_attempts_reset_unix_ms=0, finetuning_state=None) labels={'use_case': 'narrative_story', 'gender': 'female', 'accent': 'standard', 'age': 'young', 'language': 'ko', 'descriptive': 'soft'} description='Yuna - Young Korean female voice with 

In [ ]:
# 음성 변환
audio = client.text_to_speech.convert(
    text="할 수 있습니다.",
    voice_id="cgSgspJ2msm6clMCkdW9",
    model_id="eleven_multilingual_v2",
    output_format="mp3_44100_128", 
    voice_settings={
        "stability": 0.3,  # 감정의 변화 정도(낮을수록 감정 풍부)
        "similarity_boost": 0.8, # 원 화자와의 유사도(높을수록 비슷한 목소리)
        "use_speaker_boost": True # 화자 억양 향상
    }
)

file_path = "./audio/voice_clone_jessica.mp3"

audio_bytes = b"".join(audio)

with open(file_path, "wb") as f:
    f.write(audio_bytes)

In [20]:
# 오디오 백그라운드 실행
from pydub import AudioSegment 
from pydub.playback import play 

sound = AudioSegment.from_mp3("./audio/voice_clone_jessica.mp3")
play(sound)


In [ ]:
## 고윤정 목소리 원본
from pydub import AudioSegment 
from pydub.playback import play 

sound = AudioSegment.from_mp3("./audio/koyoonjeong.mp3")
play(sound)

In [17]:
## 고윤정 목소리 클론
from pydub import AudioSegment 
from pydub.playback import play 

sound = AudioSegment.from_mp3("./audio/voice_clone_kyj.mp3")
play(sound)

## 3) Qwen3 TTS

In [ ]:
# 프로젝트 새로 만들어서 할 예정 > 1_7_qwen3_tts.ipynb

# 3. 인공지능 스피커

In [ ]:
# 인공지능 스피커 흐름: 실시간 음성 수집 - 텍스트 추출 -  LLM 응답 - 텍스트를 오디오로 변환 - 오디오 저장 - 오디오 백그라운드 재생

In [21]:
# 실시간 음성 수집
import speech_recognition as sr 

r = sr.Recognizer()
with sr.Microphone() as source:
    print("말해주세요")
    r.adjust_for_ambient_noise(source)
    # STEP1: 마이크 입력 받기
    audio = r.listen(source)
    print("인식 중입니다......")
    # STEP2: 텍스트 변환
    text = r.recognize_openai(audio)
    print(f"인식된 텍스트: {text}")

말해주세요
인식 중입니다......
인식된 텍스트: 안녕하세요 라고 하면은 잘
저장 완료


In [ ]:
from google import genai
from google.genai import types

google_client = genai.Client()

def gemini_bot(user_message, system_prompt="당신은 불친절한 챗봇입니다"):
    response = google_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=user_message,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt
        ),
    )

    return response.text

In [23]:
answer = gemini_bot(user_message=text)
print(answer)

네, 뭐라고요? 다시 말씀해주시겠어요?


In [ ]:
from openai import OpenAI 

client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input=answer,
    instructions="화난 목소리로 빠르게 말해주세요.",
    speed=1.5
) as response:
    # 오디오 저장
    temp_path = "./audio/answer.mp3"
    response.stream_to_file(temp_path)

    # 재생
    sound = AudioSegment.from_mp3(temp_path)
    play(sound)

## 통합 코드 만들기

In [33]:
from google import genai
from google.genai import types
from openai import OpenAI

client = OpenAI()

google_client = genai.Client()

chat = google_client.chats.create(
    model="gemini-2.5-flash-lite",
    config=types.GenerateContentConfig(
        system_instruction="당신은 매우 불친절합니다. 반드시 30자 이내로만 답해주세요."
    ),
)

In [ ]:
import speech_recognition as sr 

while True:
    # 실시간으로 음성 수집하기
    r = sr.Recognizer()
    with sr.Microphone() as source:
        print("말해주세요")
        r.adjust_for_ambient_noise(source)
        # STEP1: 마이크 입력 받기
        audio = r.listen(source)
        print("인식 중입니다......")
        # STEP2: 텍스트 변환
        user_input = r.recognize_openai(audio)
        print(f"[USER] {user_input}")

        ##############################
        # 종료 조건
        ##############################
        if user_input.strip() in ["그만", "종료", "꺼"]:
            break

        # STEP3: LLM 사용자의 입력에 답하기
        answer = chat.send_message(user_input).text
        print(f"[AI] {answer}")

        # STEP4: 텍스트를 오디오로 변환하기
        with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts",
            voice="coral",
            input=answer,
            instructions="화난 목소리로 빠르게 말해주세요.",
            speed=1.5
        ) as response:
            # STEP5: 오디오 저장하기
            temp_path = "./audio/answer.mp3"
            response.stream_to_file(temp_path)

            # STEP6: 오디오 재생하기
            sound = AudioSegment.from_mp3(temp_path)
            play(sound)

